In [86]:
from langchain.messages import HumanMessage, SystemMessage, AIMessage
from langchain.tools import tool
from deepagents import create_deep_agent, FilesystemPermission, CompiledSubAgent
from langchain.agents import  create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from deepagents.backends import FilesystemBackend,StateBackend,StoreBackend,CompositeBackend
from langchain.chat_models import init_chat_model
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

True

In [87]:
memory = InMemorySaver()
store = InMemoryStore()

In [88]:
model = init_chat_model('openai:gpt-4')

In [89]:
from pathlib import  Path
from langchain_core.documents import Document

source_pth = Path("./files/office.txt")

docs = [Document(page_content=source_pth.read_text(encoding='utf-8'),metadata={'source':str(source_pth)})]

In [90]:
from langchain_text_splitters import  RecursiveCharacterTextSplitter

splits = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=100
)

chunks = splits.split_documents(docs)

In [91]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings()

In [92]:
from langchain_core.vectorstores import  InMemoryVectorStore

vectorstore = InMemoryVectorStore.from_documents(chunks,embedding)

In [93]:
retriever = vectorstore.as_retriever()

In [94]:
@tool
def search_document(query:str)-> str:
    """return relavant passage from the loaded document."""
    found = retriever.invoke(query)
    return "\n\n".join(doc.page_content for doc in found)

In [95]:
sub_agent_retriever = create_agent(
    model=model,
    tools=[search_document],
    system_prompt=(
        "You answer questions about the loaded document. "
        "Always use search_document to ground your answers."
    ),
)

In [96]:

# result = sub_agent_retriever.invoke({'messages': HumanMessage(content='how many people work in meridian analytics and their names?')})
# print(result['messages'][-1].content)


In [97]:
retriever_agent = CompiledSubAgent(
    name= "retriever agent",
    description = "Specilized agent for retrieving documents",
    runnable = sub_agent_retriever
)

In [98]:
@tool
def get_weather(city:str):
    """"get weather of a city"""
    return f"The weather of  {city} is 18 degrees celcius and rainy"

In [99]:
backend = CompositeBackend(
    default=StateBackend(),
    routes={
        "/memories/": StoreBackend(namespace=lambda _rt: ("use_one",)),
        "/files/": FilesystemBackend(root_dir='./files/',virtual_mode=True),
        "/skills/": FilesystemBackend(root_dir='./skills/', virtual_mode=True),
    }
)


In [100]:
agent = create_deep_agent(
    model=model,
    tools=[get_weather],
    system_prompt= "Use retriever_agent subagent to answer questions about the loaded document.",
    checkpointer=memory,
    store=store,
    backend=backend,
    skills=["/skills/"],
    subagents = [retriever_agent],
    permissions=[
    FilesystemPermission(
        operations=["write"],
        paths=["/skills/**"],
        mode="deny",
        ),
    ],
)

In [101]:
# agent = create_agent(
#     model=model,
#     tools=[get_weather],
#     system_prompt="You are a friendly assistant",
#     checkpointer=memory
# )

In [102]:
config = {'configurable':{'thread_id':'1'}}
config2 = {'configurable':{'thread_id':'2'}}

In [103]:
result = agent.invoke({'messages':HumanMessage(content="who is Priya, what position does she have, and what about Sarah, Marcus, and Lisa?")},config=config2)
print(result['messages'][-1].content)

Here are the positions and relevant details for the mentioned individuals:

1. Priya Raman is the Chief Executive Officer at Meridian Analytics. She co-founded the company with Daniel Okoye. Priya still codes on Friday afternoons and drinks exactly two cups of black coffee each day. At the end of every all-hands meeting, she uses the phrase, "Ship something you'd be proud to explain."

2. Sarah Whitfield leads the Engineering department at Meridian. She manages three teams: Platform, Ingest, and Frontend. Sarah joined the company as the seventh employee in 2019 and was promoted to Director of Engineering in 2022. She keeps a rubber duck named Gordon on her desk, claiming it has caught more bugs than any linter.

3. Marcus Delgado serves as the Principal Data Scientist at the firm. He built the initial demand-forecasting model that evolved into PulseGrid's core engine. Marcus was the one who discovered that the model's accuracy dropped annually in late November due to a Thanksgiving dem